**Objective:** Use two-way tables to calculate probabilities and statistically test if variables are independent.



* **Task 1: Universal (The Basics)**
  * *Instructions:* Using your Pet_Owner dataset, calculate the marginal distribution for `Pet_Owner`. Write a function `probability_pet_given_gender(gender)` to calculate the conditional probability.
  * *Acceptance Criteria:* Marginal percentages sum to 100%. The function correctly filters data and returns a percentage.
*

In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import random

In [3]:
gender = ['Male','Female']
pet_own = ["Yes","No"]
n_people = 100

dct = {'Gender':[],
       'Owns_Pet':[]}

for n in range(n_people):
    dct['Gender'].append(random.choice(gender))
    if dct['Gender'][-1] == 'Female':
        dct['Owns_Pet'].append(random.choices(pet_own,weights=[0.60,0.40])[0])
    elif dct['Gender'][-1] == 'Male':
        dct['Owns_Pet'].append(random.choices(pet_own,weights=[0.5,0.5])[0])

In [4]:
df_pet_owner = pd.DataFrame(dct)
df_pet_owner.value_counts()

Gender  Owns_Pet
Female  Yes         35
Male    No          26
        Yes         20
Female  No          19
Name: count, dtype: int64

In [5]:
pd.crosstab(df_pet_owner['Gender'],df_pet_owner['Owns_Pet'],normalize='index')

Owns_Pet,No,Yes
Gender,,
Female,0.351852,0.648148
Male,0.565217,0.434783


In [6]:
def probability_pet_given_gender(gender):
    gender_total_count = df_pet_owner.loc[df_pet_owner['Gender'] == gender.capitalize(), 'Gender'].count()
    pet_owner_count_for_gender = len(df_pet_owner.loc[(df_pet_owner['Gender'] == gender.capitalize()) & (df_pet_owner['Owns_Pet'] == 'Yes')])
    non_pet_owner_count_for_gender = len(df_pet_owner.loc[(df_pet_owner['Gender'] == gender.capitalize()) & (df_pet_owner['Owns_Pet'] == 'No')])
    probability_owning_pet_given_gender = (pet_owner_count_for_gender / gender_total_count) * 100
    probability_not_owning_pet_given_gender = (non_pet_owner_count_for_gender / gender_total_count) * 100
    return probability_owning_pet_given_gender, probability_not_owning_pet_given_gender

probability_owning_pet_given_gender, probability_not_owning_pet_given_gender = probability_pet_given_gender('female')
marginal_distribution = df_pet_owner['Owns_Pet'].value_counts(normalize=True) * 100

print(f"The estimated probability of owning a pet for the female group is -> {round(probability_owning_pet_given_gender, 2)}%")
print(f"The estimated probability of not owning a pet for the female group is -> {round(probability_not_owning_pet_given_gender, 2)}%")
print(f"Check whether the conditional probabilities sum to 100% -> {round(probability_owning_pet_given_gender + probability_not_owning_pet_given_gender, 2)}%")
print(f"Marginal distribution of the 'Owns_Pet' column:\n{marginal_distribution}")
print(marginal_distribution.sum())

The estimated probability of owning a pet for the female group is -> 64.81%
The estimated probability of not owning a pet for the female group is -> 35.19%
Check whether the conditional probabilities sum to 100% -> 100.0%
Marginal distribution of the 'Owns_Pet' column:
Owns_Pet
Yes    55.0
No     45.0
Name: proportion, dtype: float64
100.0


* **Task 2: Capstone (The Antifraud Logic)**
  * *Instructions:* Using your Credit Card dataset, calculate the marginal distribution for `Is_Fraud` (base rate). Write a function `probability_fraud_given_location(location)`.
  * *Acceptance Criteria:* `probability_fraud_given_location('International')` returns a percentage significantly higher than the base rate.


In [7]:
### Artifical Dataset,Synthetic Dataset
dct_transactions = {
    'Transaction_Location':[],
    'Is_Fraud':[]
}

locations = ['Local','International']
fraud = ['Yes','No']

n_transactions = 1000
for n in range(n_transactions):
    dct_transactions['Transaction_Location'].append(random.choices(locations,weights=[0.9,0.1])[0])
    if dct_transactions['Transaction_Location'][-1] == 'International':
        dct_transactions['Is_Fraud'].append(random.choices(fraud,weights=[0.7,0.3])[0])
    elif dct_transactions['Transaction_Location'][-1] == 'Local':
        dct_transactions['Is_Fraud'].append(random.choices(fraud,weights=[0.05,0.95])[0])

df_transactions = pd.DataFrame(dct_transactions)

In [8]:
given_location = 'International'
def probability_fraud_given_location(location):
    location_total_count = df_transactions.loc[df_transactions['Transaction_Location'] == location.capitalize(), 'Transaction_Location'].count()
    fraud_at_location_count = len(df_transactions.loc[(df_transactions['Transaction_Location'] == location.capitalize()) & (df_transactions['Is_Fraud'] == 'Yes')])
    no_fraud_at_location_count = len(df_transactions.loc[(df_transactions['Transaction_Location'] == location.capitalize()) & (df_transactions['Is_Fraud'] == 'No')])
    fraud_probability_given_location = (fraud_at_location_count / location_total_count) * 100
    no_fraud_probability_given_location = (no_fraud_at_location_count / location_total_count) * 100
    return fraud_probability_given_location, no_fraud_probability_given_location

fraud_probability_for_location = probability_fraud_given_location('International')[0]
no_fraud_probability_for_location = probability_fraud_given_location('International')[1]
base_rate  = (df_transactions['Is_Fraud']=='Yes').mean() * 100
marginal = df_transactions['Is_Fraud'].value_counts(normalize=True) * 100

print(f"Probability of a transaction being fraudulent given the location is [{given_location}]-> {fraud_probability_for_location.round(2)}%")
print(f"Probability of a transaction not being fraudulent given the location is [{given_location}]-> {no_fraud_probability_for_location.round(2)}%")
print(f"Base rate: {base_rate:.2f}%")
print(f"Marginal distribution of the 'Is_Fraud' column:\n{marginal}")


Probability of a transaction being fraudulent given the location is [International]-> 77.42%
Probability of a transaction not being fraudulent given the location is [International]-> 22.58%
Base rate: 12.00%
Marginal distribution of the 'Is_Fraud' column:
Is_Fraud
No     88.0
Yes    12.0
Name: proportion, dtype: float64


In [9]:
print(probability_fraud_given_location('International')[0].round(2))   # should be ~70%
print(probability_fraud_given_location('Local')[0].round(2))           # should be ~5%
print(base_rate.round(2))                                              # ~11.5%

77.42
5.29
12.0


* **Task 3: Hidden Concept (Testing for Independence)**
  * *Instructions:* Two variables are **independent** if P(A|B) = P(A). Write a Python function `check_independence(dataset, var1, var2)` that calculates the marginal probability of var1, and the conditional probability of var1 given var2. Print whether they are "Independent" or "Associated".
  * *Acceptance Criteria:* When you run `check_independence` on Gender vs Pet_Owner (if you made the probabilities identical), it prints "Independent". When you run it on Location vs Fraud, it prints "Associated".

In [10]:
import pandas as pd
import random

random.seed(42)

# --- Dataset 1: Gender vs Pet_Owner — INDEPENDENT ---
# Both genders use IDENTICAL weights → P(Pet|Female) == P(Pet|Male) == P(Pet)
gender = ['Male', 'Female']
pet_options = ['Yes', 'No']

n_people = 2000   # larger n reduces sampling noise

dct_independent = {'Gender': [], 'Owns_Pet': []}

for _ in range(n_people):
    g = random.choice(gender)
    dct_independent['Gender'].append(g)
    # Same 50/50 weights for both genders — gender has no effect on pet ownership
    dct_independent['Owns_Pet'].append(random.choices(pet_options, weights=[0.5, 0.5])[0])

df_independent = pd.DataFrame(dct_independent)

print(pd.crosstab(df_independent['Gender'], df_independent['Owns_Pet'], normalize='index') * 100)

Owns_Pet         No        Yes
Gender                        
Female    50.146915  49.853085
Male      49.744637  50.255363


In [27]:
def check_independence(df:pd.DataFrame,var1,var2):
    gaps = []
    marginal_val = df.loc[df[var1] == 'Yes',var1].count()/df.shape[0] * 100
    for v in df[var2].unique():
        conditional = (df.loc[(df[var1] == 'Yes') & (df[var2]==v)].shape[0])/df.loc[df[var2] == v,var2].count() * 100
        gaps.append(abs(conditional-marginal_val))
    tolerance = 5
    if all(g < tolerance for g in gaps):
        print("Independent")
    else:
        print("Associated")


    
check_independence(df_pet_owner,'Owns_Pet','Gender')

Associated


***

# Part 2 — The Concepts (Medieval Fantasy Setting)

## The Setup
You are the Royal Statistician to King Aldric of Eldoria. The kingdom has three regions — Northlands, Southlands, and Midlands — and a chronic dragon problem. Every month, villages report whether they were attacked.

The King has two questions:
1. What's the overall dragon attack rate? (**Marginal**)
2. Does the region matter? (**Independence test**)

**Your dataframe:**

| Village | Region | Dragon_Attack |
| :--- | :--- | :--- |
| A | Northlands | Yes |
| B | Southlands | No |
| C | Midlands | Yes |
| ... | ... | ... |

* `var1` = Dragon_Attack (positive class = 'Yes')
* `var2` = Region

---

## Marginal Probability — The Kingdom-Wide Picture
$$ P(\text{Dragon\_Attack} = \text{Yes}) = \frac{\text{number of attacked villages}}{\text{total villages}} $$

Suppose Eldoria has 1,000 villages and 300 were attacked. Then:
$$ \text{marginal} = \frac{300}{1000} = 30\% $$

> **Real-world translation:** This is your base rate — the prior probability of an attack before you know anything about the village. In business, it's *"what fraction of users churn overall?"* or *"what fraction of transactions are fraudulent?"* before you look at any features.
> 
> **Fantasy translation:** The King now knows that if he randomly teleports to a village, there's a 30% chance it was attacked this year. Useful, but doesn't tell him where to send guards.

---

## Conditional Probability — Does Region Change the Picture?
$$ P(\text{Attack} \mid \text{Region} = \text{Northlands}) = \frac{\text{attacked villages in Northlands}}{\text{total villages in Northlands}} $$

Suppose the Northlands have 200 villages and 140 were attacked:
$$ P(\text{Attack} \mid \text{Northlands}) = \frac{140}{200} = 70\% $$

Similarly:

| Region | $P(\text{Attack} \mid \text{Region})$ |
| :--- | :--- |
| Northlands | 70% |
| Southlands | 5% |
| Midlands | 15% |

> **Real-world translation:** This is exactly what a machine learning model does when it splits on a feature. $P(\text{churn} \mid \text{Premium})$ vs $P(\text{churn} \mid \text{Free})$. $P(\text{fraud} \mid \text{International})$ vs $P(\text{fraud} \mid \text{Local})$.
> 
> **Fantasy translation:** The King now knows the Northlands are a war zone. Knowing the region changed his belief about whether a village was attacked. The region is a **discriminative feature**.

---

## Independence — "Knowing B tells me nothing about A"
The formal definition:
$$ A \perp B \iff P(A \mid B) = P(A) $$

**Read it in plain English:** The conditional equals the marginal for every value of $B$.

Now imagine a different Eldoria where the data looks like this:

| Region | $P(\text{Attack} \mid \text{Region})$ |
| :--- | :--- |
| Northlands | 31% |
| Southlands | 28% |
| Midlands | 30% |
| **Marginal** | **30%** |

Every conditional is $\approx 30\%$ — the same as the marginal. So:
* Learning a village is in the Northlands $\rightarrow$ belief about attack stays at ~30%.
* Learning it's in the Southlands $\rightarrow$ belief stays at ~30%.
* Learning it's in the Midlands $\rightarrow$ belief stays at ~30%.

The region gave you **zero information**. Dragon attacks happen uniformly across the kingdom. Region and attack are **independent**.

> **Real-world translation:** If $P(\text{churn} \mid \text{Premium}) == P(\text{churn} \mid \text{Free})$, then `Subscription_Tier` is useless as a feature — the model can't use it to separate churners from non-churners. You'd drop it.
> 
> **Fantasy translation:** The King does not need to station more guards in any particular region. He saves the treasury.

---

## Tolerance — The Reality of Noise
Here's the twist. In a mathematically perfect world, $P(A \mid B) = P(A)$ exactly. But you never see the true probabilities — you see a **sample**.

If Eldoria truly has a 30% attack rate in every region, a sample of 1,000 villages might still show:

| Region | Observed $P(\text{Attack} \mid \text{Region})$ |
| :--- | :--- |
| Northlands | 31.2% |
| Southlands | 28.4% |
| Midlands | 30.6% |

The gaps from the 30% marginal are 1.2, 1.6, and 0.6 percentage points. Not zero — but they should be attributed to **random chance**, not a real regional effect. 

If the code used `==`, it would print "Associated" — a wrong verdict, because in reality the region doesn't matter.

This is where **tolerance** comes in:
$$ \text{If } |P(A \mid B) - P(A)| < \epsilon \text{ for all values of } B, \text{ declare independence.} $$

A tolerance of `5` means *"a gap under 5 percentage points is small enough to be sampling noise."* It's the statistical equivalent of saying "close enough."

> **Real-world translation:** This is the same instinct that leads to p-values, confidence intervals, and effect sizes. You never compare observed values with strict equality — you always allow a margin for noise. Without tolerance, no real-world dataset would ever show "independence."
> 
> **Fantasy translation:** The Royal Statistician tells the King: *"The region-based gaps are 1–2 percentage points, which is within the noise of a 1,000-village sample. I cannot conclude the regions differ. Send no extra guards."* The tolerance prevents the King from wasting gold chasing phantom signals.

**🎲 Tavern-dice mini-example:** 
Two dice are thrown 100 times each. Both are fair (true $P(6) = 1/6 = 16.7\%$). But the red die shows 6 eighteen times (18%) and the blue die fifteen times (15%). Gaps: 1.3 and 1.7 percentage points. Within tolerance $\rightarrow$ *"Independent — both dice are fair."* Without tolerance, you'd falsely accuse the blue die of cheating.

---

## Why Compare Marginal to Conditional?
Because that is literally the **definition of independence**. It's not an arbitrary test — it's the mathematical identity that distinguishes *"B is informative about A"* from *"B is not informative about A."*

| If... | Then... | Means... |
| :--- | :--- | :--- |
| Conditional $\approx$ Marginal | **Independent** | Knowing B doesn't change belief about A |
| Conditional $\neq$ Marginal | **Associated** | Knowing B changes belief about A |

The size of the gap is the **effect size**. Tiny gap $\rightarrow$ feature is useless. Huge gap $\rightarrow$ feature is highly predictive.

***

# Part 3 — Why This Matters for Your Data Science Career

This exercise is a simplified version of **feature selection**, one of the most important steps in any ML pipeline.

### 1. Feature Engineering
Before feeding a feature into a model, you check: does this feature carry signal about the target? If $P(\text{Target} \mid \text{Feature}) \approx P(\text{Target})$ for every value of the feature, the feature is uninformative — **drop it**. You just saved the model time and reduced overfitting risk.

### 2. Fairness and Bias Auditing
If `Region` in a loan-approval dataset has $P(\text{Approved} \mid \text{Region})$ varying widely across regions, that's evidence of bias. Independence testing is the first line of defense in algorithmic fairness audits.

### 3. A/B Testing
Replace `Region` with `Treatment_Group` (Control / Variant) and `Dragon_Attack` with `Converted` (Yes / No). If $P(\text{Converted} \mid \text{Variant}) \neq P(\text{Converted} \mid \text{Control})$, your new UI changed behavior. This is the same test you'll run in every product experiment.

### 4. Fraud Detection (Your Task 2)
You already did this — the whole point of the antifraud task was that $P(\text{Fraud} \mid \text{International})$ is much greater than $P(\text{Fraud})$, so location is a valid feature for a fraud classifier.

### 5. Feature Monitoring in Production
After deployment, you re-run independence checks to detect drift. If a previously-informative feature's conditionals start matching the marginal again, its predictive power has collapsed — **time to retrain**.

---

## Summary Table

| Concept | Fantasy Meaning | Real-World Meaning |
| :--- | :--- | :--- |
| **Marginal** $P(A)$ | Overall attack rate across all regions | Base rate of churn, fraud, conversion |
| **Conditional** $P(A \mid B)$ | Attack rate within a specific region | Conversion rate within a specific user segment |
| **Independence** | Region doesn't predict attack | Feature has no signal about target |
| **Association** | Region strongly predicts attack | Feature is discriminative |
| **Tolerance** | Sampling noise threshold (1–2 pts OK) | p-value / effect-size analogue |
| **Comparison** | Gap between conditional and marginal | Quantifying feature importance |

Once you internalize this pattern, you'll see it everywhere — every `groupby().mean()` in a business dashboard is a conditional probability in disguise, and every feature-selection decision is an independence test waiting to happen.

> *And yes, before you ask: the function is now correct. Just fix the `df_pet_owner` dataset so the two genders have identical probabilities, and both acceptance criteria will pass.*

* **Task 4: Capstone (Product Analytics - Feature Adoption)**
  * *Context:* Before building a Machine Learning model to predict user churn, a Product Analyst needs to verify if a user's subscription tier affects their likelihood of adopting a new UI feature.
  * *Instructions:* Generate a synthetic dataset of 1,000 users. Assign `Subscription_Tier` (20% 'Premium', 80% 'Free'). Assign `Adopted_New_Feature` ('Yes' or 'No'), but bias the data so 'Premium' users are 3x more likely to adopt the feature than 'Free' users. Build a two-way table. Use your `check_independence` function to evaluate if `Subscription_Tier` and `Adopted_New_Feature` are associated.
  * *Acceptance Criteria:* The script generates the biased dataset, builds the frequency table, and correctly prints "Associated" (because the conditional probability of adopting the feature given Premium status will be significantly higher than the marginal probability of adopting the feature overall). This proves to the ML team that `Subscription_Tier` is a valid categorical feature to include in their churn prediction model.
  